In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Native Python 5-Fold OOF Stacking Multinomial Logistic Pipeline (`models/train_oof_logistic_regression_stacking.ipynb`)

This notebook trains the **entire 5-Fold OOF Stacking Pipeline natively in Python scikit-learn & LightGBM**:

### Data Cleaning & Stratification
- Filters all observations where ESI target level or any required vital sign feature is NA.
- Uses exact stratified train/test partitioning to guarantee balanced class distributions across ESI 1 through 5.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, OMIT NaNs & Partition Stratified Split
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
raw_esi <- as.character(raw_df[[target_col_name]])
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)
# STRICT CLEANING: Remove any rows with NA features or NA ESI target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))
test_size <- config$training$test_size
in_train  <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
train_df_clean <- df_master[in_train, ]
test_df_clean  <- df_master[-in_train, ]
train_mat_export <- as.matrix(cbind(train_df_clean[, 1:15], target = train_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:15],  target = test_df_clean$target_num))
cat(sprintf("Data Cleaned & Split: Train=%d, Test=%d (Zero NA Rows, Valid ESI 1..5)\n", nrow(train_mat_export), nrow(test_mat_export)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Native Python Feature Engineering, 5-Fold CV OOF Stacking & Model Training
# ---------------------------------------------------------
import os
import pickle
import numpy as np
import pandas as pd
from rpy2.robjects import r
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)
raw_mat_tr = train_mat_in[:, :15]
y_train    = train_mat_in[:, 15].astype(int)
raw_mat_ts = test_mat_in[:, :15]
y_test     = test_mat_in[:, 15].astype(int)
def build_38_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 38), dtype=np.float64)
    X[:, :15] = raw_mat
    
    t_hr = raw_mat[:, 3]; t_sbp = raw_mat[:, 4]; t_rr = raw_mat[:, 5]; t_o2 = raw_mat[:, 6]
    pulse_min = raw_mat[:, 7]; resp_min = raw_mat[:, 8]; spo2_min = raw_mat[:, 9]; sbp_min = raw_mat[:, 10]
    pulse_max = raw_mat[:, 11]; resp_max = raw_mat[:, 12]; spo2_max = raw_mat[:, 13]; sbp_max = raw_mat[:, 14]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    X[:, 15] = (t_o2 < 90).astype(float)
    X[:, 16] = ((t_o2 > 90) & (t_o2 < 94)).astype(float)
    X[:, 17] = (t_rr < 10).astype(float)
    X[:, 18] = (t_rr > 30).astype(float)
    X[:, 19] = (t_sbp <= 90).astype(float)
    X[:, 20] = (t_sbp > 220).astype(float)
    X[:, 21] = (t_hr < 40).astype(float)
    X[:, 22] = ((t_hr > 40) & (t_hr < 60)).astype(float)
    X[:, 23] = (t_hr > 150).astype(float)
    X[:, 24] = ((t_hr > 100) & (t_hr < 150)).astype(float)
    X[:, 25] = hr_rng; X[:, 26] = rr_rng; X[:, 27] = spo2_rng; X[:, 28] = sbp_rng
    X[:, 29] = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 30] = t_hr - hr_rng
    X[:, 31] = t_sbp - sbp_rng
    X[:, 32] = t_rr - rr_rng
    X[:, 33] = t_o2 - spo2_rng
    X[:, 34] = t_o2 / np.where(t_rr == 0, 1.0, t_rr)
    X[:, 35] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max)
    X[:, 36] = hr_rng / (t_hr + 1.0)
    X[:, 37] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0
    return X
X_train_raw = build_38_feature_matrix(raw_mat_tr)
X_test_raw  = build_38_feature_matrix(raw_mat_ts)
feature_names = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif'
]
cont_cols_idx = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_test  = X_test_raw.copy()
X_train[:, cont_cols_idx] = scaler.fit_transform(X_train_raw[:, cont_cols_idx])
X_test[:, cont_cols_idx]  = scaler.transform(X_test_raw[:, cont_cols_idx])
# Custom Fast NumPy SMOTE Helper
def numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]))
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])
# 5-Fold Cross-Validation OOF Probability Generation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((len(X_train), 5))
test_probs_folds = np.zeros((len(X_test), 5))
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr_k, y_tr_k = X_train[tr_idx], y_train[tr_idx]
    X_vl_k, y_vl_k = X_train[val_idx], y_train[val_idx]
    
    # L1 (ESI 1 vs Rest)
    X_sm1, y_sm1 = numpy_smote(X_tr_k, (y_tr_k == 1).astype(int), seed=42 + fold)
    clf_l1 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
    clf_l1.fit(X_sm1, y_sm1, eval_set=[(X_vl_k, (y_vl_k == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
    p1_k  = clf_l1.predict_proba(X_vl_k)[:, 1]
    p1_ts = clf_l1.predict_proba(X_test)[:, 1]
    
    # L2 (ESI 2,3 vs ESI 4,5)
    m2_tr = (y_tr_k != 1); m2_vl = (y_vl_k != 1)
    X_sm2, y_sm2 = numpy_smote(X_tr_k[m2_tr], np.isin(y_tr_k[m2_tr], [2, 3]).astype(int), seed=42 + fold)
    clf_l2 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
    clf_l2.fit(X_sm2, y_sm2, eval_set=[(X_vl_k[m2_vl], np.isin(y_vl_k[m2_vl], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
    p2_k  = clf_l2.predict_proba(X_vl_k)[:, 1]
    p2_ts = clf_l2.predict_proba(X_test)[:, 1]
    
    # L3A (ESI 2 vs ESI 3)
    m3a_tr = np.isin(y_tr_k, [2, 3]); m3a_vl = np.isin(y_vl_k, [2, 3])
    X_sm3a, y_sm3a = numpy_smote(X_tr_k[m3a_tr], (y_tr_k[m3a_tr] == 2).astype(int), seed=42 + fold)
    clf_l3a = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
    clf_l3a.fit(X_sm3a, y_sm3a, eval_set=[(X_vl_k[m3a_vl], (y_vl_k[m3a_vl] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
    p3a_k  = clf_l3a.predict_proba(X_vl_k)[:, 1]
    p3a_ts = clf_l3a.predict_proba(X_test)[:, 1]
    
    # L3B (ESI 4 vs ESI 5)
    m3b_tr = np.isin(y_tr_k, [4, 5]); m3b_vl = np.isin(y_vl_k, [4, 5])
    X_sm3b, y_sm3b = numpy_smote(X_tr_k[m3b_tr], (y_tr_k[m3b_tr] == 4).astype(int), seed=42 + fold)
    clf_l3b = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
    clf_l3b.fit(X_sm3b, y_sm3b, eval_set=[(X_vl_k[m3b_vl], (y_vl_k[m3b_vl] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
    p3b_k  = clf_l3b.predict_proba(X_vl_k)[:, 1]
    p3b_ts = clf_l3b.predict_proba(X_test)[:, 1]
    
    oof_probs[val_idx, 0] = p1_k
    oof_probs[val_idx, 1] = (1 - p1_k) * p2_k * p3a_k
    oof_probs[val_idx, 2] = (1 - p1_k) * p2_k * (1 - p3a_k)
    oof_probs[val_idx, 3] = (1 - p1_k) * (1 - p2_k) * p3b_k
    oof_probs[val_idx, 4] = (1 - p1_k) * (1 - p2_k) * (1 - p3b_k)
    
    test_probs_folds[:, 0] += p1_ts / 5.0
    test_probs_folds[:, 1] += ((1 - p1_ts) * p2_ts * p3a_ts) / 5.0
    test_probs_folds[:, 2] += ((1 - p1_ts) * p2_ts * (1 - p3a_ts)) / 5.0
    test_probs_folds[:, 3] += ((1 - p1_ts) * (1 - p2_ts) * p3b_ts) / 5.0
    test_probs_folds[:, 4] += ((1 - p1_ts) * (1 - p2_ts) * (1 - p3b_ts)) / 5.0
# Fit Multinomial Logistic Regression Meta-Learner
meta_logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_logreg.fit(oof_probs, y_train)
# Train Production Sub-Models on 100% Train Data with Early Stopping
X_tr_p, X_vl_p, y_tr_p, y_vl_p = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)
X_sm1_p, y_sm1_p = numpy_smote(X_tr_p, (y_tr_p == 1).astype(int))
l1_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l1_prod.fit(X_sm1_p, y_sm1_p, eval_set=[(X_vl_p, (y_vl_p == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
m2_tr_p = (y_tr_p != 1); m2_vl_p = (y_vl_p != 1)
X_sm2_p, y_sm2_p = numpy_smote(X_tr_p[m2_tr_p], np.isin(y_tr_p[m2_tr_p], [2, 3]).astype(int))
l2_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l2_prod.fit(X_sm2_p, y_sm2_p, eval_set=[(X_vl_p[m2_vl_p], np.isin(y_vl_p[m2_vl_p], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
m3a_tr_p = np.isin(y_tr_p, [2, 3]); m3a_vl_p = np.isin(y_vl_p, [2, 3])
X_sm3a_p, y_sm3a_p = numpy_smote(X_tr_p[m3a_tr_p], (y_tr_p[m3a_tr_p] == 2).astype(int))
l3a_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3a_prod.fit(X_sm3a_p, y_sm3a_p, eval_set=[(X_vl_p[m3a_vl_p], (y_vl_p[m3a_vl_p] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
m3b_tr_p = np.isin(y_tr_p, [4, 5]); m3b_vl_p = np.isin(y_vl_p, [4, 5])
X_sm3b_p, y_sm3b_p = numpy_smote(X_tr_p[m3b_tr_p], (y_tr_p[m3b_tr_p] == 4).astype(int))
l3b_prod = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3b_prod.fit(X_sm3b_p, y_sm3b_p, eval_set=[(X_vl_p[m3b_vl_p], (y_vl_p[m3b_vl_p] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])
# Export Bundle to deploy/py_oof_stacking_bundle.pkl
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
os.makedirs(deploy_dir, exist_ok=True)
bundle_data = {
    'l1_prod': l1_prod,
    'l2_prod': l2_prod,
    'l3a_prod': l3a_prod,
    'l3b_prod': l3b_prod,
    'meta_logreg': meta_logreg,
    'scaler_means': scaler.mean_,
    'scaler_sds': scaler.scale_,
    'cont_cols_idx': cont_cols_idx,
    'feature_names': feature_names
}
with open(os.path.join(deploy_dir, 'py_oof_stacking_bundle.pkl'), 'wb') as f:
    pickle.dump(bundle_data, f)
print(f"Python Native OOF Stacking Bundle saved to: {os.path.join(deploy_dir, 'py_oof_stacking_bundle.pkl')}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Holdout Test Set Evaluation & Detailed Per-Class Breakdown
# ---------------------------------------------------------
preds_unweighted  = np.argmax(test_probs_folds, axis=1) + 1
preds_meta_logreg = meta_logreg.predict(test_probs_folds)
probs_meta_logreg = meta_logreg.predict_proba(test_probs_folds)
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)
df_unw    = get_per_class_breakdown(y_test, preds_unweighted,  test_probs_folds,  '3Tier_Soft_Pipeline_Unweighted')
df_logreg = get_per_class_breakdown(y_test, preds_meta_logreg,probs_meta_logreg,  'OOF_Stacking_Multinomial_Logistic_Regression')
full_report_df = pd.concat([df_unw, df_logreg], ignore_index=True)
print("========================================================================================")
print("   HOLDOUT TEST SET REPORT: PYTHON NATIVE OOF MULTINOMIAL LOGISTIC META-LEARNER")
print("========================================================================================")
print(full_report_df.to_string(index=False))
print("========================================================================================\n")
reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)
full_report_df.to_csv(os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report.csv')}")